In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("performance_log.csv")
df["grid"] = df["grid"].astype(int)

# Keep matrix-scale runs (drop smoke tests / tiny grids)
bench = df[(df["count"] >= 100_000) & (df["grid"] >= 4096)].copy()
bench = bench.sort_values(["feature", "count", "grid", "datetime"])
# If the same (feature, count, grid) was run more than once, keep the latest
bench = bench.drop_duplicates(subset=["feature", "count", "grid"], keep="last")

print(bench.groupby("feature").size())
bench.head()

In [ ]:
def plot_time_vs_count(data, feature, time_col="rast_s"):
    """Multi-line log-log plot: time vs count, one line per grid size."""
    subset = data[data["feature"] == feature].sort_values("count")
    if subset.empty:
        print(f"No rows for feature={feature!r}")
        return

    counts = sorted(subset["count"].unique())

    fig, ax = plt.subplots(figsize=(9, 5.5))

    # vertical guides at each count
    for c in counts:
        ax.axvline(c, color="0.35", alpha=0.85, linewidth=1.6, zorder=0)

    for grid, group in subset.groupby("grid"):
        group = group.sort_values("count")
        ax.plot(
            group["count"],
            group[time_col],
            marker="o",
            label=f"{grid} x {grid}",
            zorder=2,
        )
        # label to the right of each point (minutes)
        for _, row in group.iterrows():
            mins = row[time_col] / 60.0
            ax.annotate(
                f"{mins:.2f} min",
                (row["count"], row[time_col]),
                textcoords="offset points",
                xytext=(8, 0),
                va="center",
                ha="left",
                fontsize=7,
                zorder=3,
            )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xticks(counts)
    ax.set_xticklabels([f"{c:.0e}" for c in counts], rotation=30, ha="right")
    ax.set_xlabel("Count")
    ax.set_ylabel("Rasterisation time (s)")
    ax.set_title(f"{feature.capitalize()}: time vs count by resolution")
    ax.legend(title="Grid")
    ax.grid(True, which="both", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.show()


plot_time_vs_count(bench, "points")
plot_time_vs_count(bench, "lines")


## Polygons (full matrix)

From `performance_log_poly.csv` (headerless export from the polygon matrix run).


In [ ]:
cols = [
    "datetime", "feature", "count", "grid",
    "gen_s", "rast_s", "enc_s", "total_s", "pixels_on", "optimised",
]
poly = pd.read_csv("performance_log_poly.csv", header=None, names=cols)
poly["grid"] = poly["grid"].astype(int)
poly = poly.sort_values(["count", "grid"])

display(poly[["count", "grid", "gen_s", "rast_s", "enc_s", "total_s", "pixels_on"]])
plot_time_vs_count(poly, "polygons")


In [ ]:
# Compare features at 4096² (points/lines from main log, polygons from poly matrix)
cmp = pd.concat([
    bench[bench["feature"].isin(["points", "lines"])],
    poly,
], ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 5))
for feature, group in cmp[cmp["grid"] == 4096].groupby("feature"):
    group = group.sort_values("count")
    ax.plot(group["count"], group["rast_s"], marker="o", label=feature)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Count")
ax.set_ylabel("Rasterisation time (s)")
ax.set_title("4096 x 4096: rasterisation time by feature")
ax.legend()
ax.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## Optimised vs baseline (boxplot)
Rows 93-112 of performance_log.csv: polygons, count=1000, grid=16384.


In [ ]:
# Side-by-side box plot: rasterisation time for not optimised vs optimised
opt_cmp = pd.read_csv("performance_log.csv").iloc[91:111].copy()
opt_cmp["optimised"] = pd.to_numeric(opt_cmp["optimised"], errors="coerce").astype(int)
opt_cmp["version"] = opt_cmp["optimised"].map({0: "not optimised", 1: "optimised"})
# keep left-to-right order
opt_cmp["version"] = pd.Categorical(
    opt_cmp["version"],
    categories=["not optimised", "optimised"],
    ordered=True,
)

fig, ax = plt.subplots(figsize=(6, 5))
opt_cmp.boxplot(column="rast_s", by="version", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Rasterisation time (s)")
ax.set_title("Polygons 1000 count, 16384 grid")
plt.suptitle("")
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

display(opt_cmp.groupby("version", observed=True)["rast_s"].agg(["count", "mean", "std", "min", "max"]))
